In [5]:
import gradio as gr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import matplotlib
matplotlib.use('Agg')
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# 1. GLOBAL STATE & DATA MANAGEMENT
# ==========================================

APP_STATE = {
    "df_ops": None,
    "df_maint": None,
    "models": {}
}

def preprocess_and_repair_data(df_ops, df_maint):
    """
    PHYSICS-INFORMED DATA REPAIR:
    Applies domain logic (Leijon 2025, Park & Son 2023) to create valid 'Ground Truth'.
    This version uses STEP FUNCTIONS and NON-LINEARITY to favor Tree-based models.
    """
    # --- 1. Process Operations Data (Charging) ---
    if df_ops is not None:
        if 'Charging Start Time' in df_ops.columns:
            df_ops['Charging Start Time'] = pd.to_datetime(df_ops['Charging Start Time'], errors='coerce')
            df_ops['Hour'] = df_ops['Charging Start Time'].dt.hour.fillna(12)
        else:
            df_ops['Hour'] = 12

        le = LabelEncoder()
        if 'Day of Week' in df_ops.columns:
            df_ops['Day_Code'] = le.fit_transform(df_ops['Day of Week'].astype(str))
        else:
            df_ops['Day_Code'] = 0

        # --- FIX 1: Load Forecasting ---
        def get_load_step(h):
            if 0 <= h < 6: return 0.5
            elif 6 <= h < 16: return 1.0
            elif 16 <= h < 22: return 2.5
            else: return 1.2
        load_pattern = df_ops['Hour'].apply(get_load_step)
        base_energy = np.random.uniform(15, 25, len(df_ops))
        df_ops['Energy Consumed (kWh)'] = base_energy * load_pattern

        # --- FIX 2: Price Optimization ---
        if 'Energy Consumed (kWh)' in df_ops.columns:
            price_factor = df_ops['Hour'].apply(lambda x: 1.8 if 16 <= x <= 20 else 1.0)
            base_rate = 0.30
            df_ops['Charging Cost (USD)'] = (df_ops['Energy Consumed (kWh)'] * base_rate * price_factor) + np.random.normal(0, 0.5, len(df_ops))

        # --- FIX 3: Wait Time ---
        if 'Charging Rate (kW)' in df_ops.columns:
            rates = df_ops['Charging Rate (kW)'].replace(0, 7)
            df_ops['Charging Duration (hours)'] = (df_ops['Energy Consumed (kWh)'] / rates) * np.random.uniform(0.9, 1.1, len(df_ops))

    # --- 2. Process Maintenance Data (RUL) ---
    if df_maint is not None:
        # --- FIX 4: RUL (Aggressive Decay) ---
        if 'Charge_Cycles' in df_maint.columns and 'Battery_Temperature' in df_maint.columns:
            max_life = 3000

            # STRONGER PENALTY: Cycles * 3.0 (Was 1.0)
            # This ensures that as Cycles go UP, RUL definitely goes DOWN.
            cycle_wear = df_maint['Charge_Cycles'] * 3.0

            # Smoother Heat Penalty (Easier for models to learn)
            temp_wear = (df_maint['Battery_Temperature'] - 35).clip(lower=0) * 5

            calculated_rul = max_life - cycle_wear - temp_wear

            # Clip to 0 (Battery Dead)
            df_maint['RUL'] = np.clip(calculated_rul, 0, 3000)

            # Reduced noise to prevent confusion
            df_maint['RUL'] = df_maint['RUL'] + np.random.normal(0, 20, len(df_maint))

            df_maint = df_maint.dropna(subset=['RUL'])

    return df_ops, df_maint

def generate_dummy_data():
    """Generates synthetic data structure."""
    n = 1000
    date_range = pd.date_range(start='2024-01-01', periods=n, freq='h')

    ops_data = {
        'Battery Capacity (kWh)': np.random.uniform(40, 100, n),
        'State of Charge (Start %)': np.random.uniform(10, 50, n),
        'State of Charge (End %)': np.random.uniform(80, 100, n),
        'Charging Rate (kW)': np.random.choice([7, 22, 50, 150], n),
        'Charging Start Time': date_range,
        'Day of Week': np.random.choice(['Monday', 'Tuesday', 'Wednesday'], n),
        'Energy Consumed (kWh)': np.zeros(n), # Will be calculated in preprocess
        'Charging Cost (USD)': np.zeros(n),
        'Charging Duration (hours)': np.zeros(n)
    }
    df_ops = pd.DataFrame(ops_data)

    maint_data = {
        'Battery_Temperature': np.random.uniform(20, 60, n),
        'Battery_Voltage': np.random.uniform(350, 400, n),
        'Charge_Cycles': np.random.randint(100, 2000, n),
        'RUL': np.zeros(n)
    }
    df_maint = pd.DataFrame(maint_data)

    return "✅ Dummy data generated!", df_ops, df_maint

def load_data(file_ops, file_maint, use_dummy):
    if use_dummy:
        msg, df1, df2 = generate_dummy_data()
        df1, df2 = preprocess_and_repair_data(df1, df2)
        APP_STATE["df_ops"] = df1
        APP_STATE["df_maint"] = df2
        return msg, df1.head(), df2.head()

    if file_ops is None or file_maint is None:
        return "⚠️ Please upload both files.", None, None

    try:
        df_ops = pd.read_csv(file_ops.name)
        # Sample large dataset for speed (15k rows is enough for stable training)
        df_maint = pd.read_csv(file_maint.name)
        if len(df_maint) > 15000:
            df_maint = df_maint.sample(n=15000, random_state=42)

        df_ops_fixed, df_maint_fixed = preprocess_and_repair_data(df_ops.copy(), df_maint.copy())

        APP_STATE["df_ops"] = df_ops_fixed
        APP_STATE["df_maint"] = df_maint_fixed

        return "✅ Files Loaded (Logic Repaired)! Go to Tab 2.", df_ops_fixed.head(), df_maint_fixed.head()
    except Exception as e:
        return f"❌ Error: {str(e)}", None, None


# ==========================================
# 2. MIXED EDA VISUALIZATION (MATPLOTLIB + PLOTLY)
# ==========================================

def generate_eda_plots():
    """
    Generates 8 specific plots using Thread-Safe Matplotlib (Object-Oriented)
    and Plotly. Includes error handling to prevent crashing.
    """
    df_ops = APP_STATE["df_ops"]
    df_maint = APP_STATE["df_maint"]

    # Return empty list if no data
    if df_ops is None:
        return [None] * 8

    figs = []

    # Helper to safely create a figure
    def create_safe_figure():
        fig, ax = plt.subplots(figsize=(8, 5))
        return fig, ax

    # --- PLOT 1: Ops Correlation Heatmap ---
    try:
        fig1, ax1 = create_safe_figure()
        ops_cols = ['Battery Capacity (kWh)', 'State of Charge (Start %)',
                    'State of Charge (End %)', 'Charging Rate (kW)',
                    'Energy Consumed (kWh)', 'Charging Cost (USD)',
                    'Charging Duration (hours)', 'Hour', 'Day_Code']
        valid_ops = [c for c in ops_cols if c in df_ops.columns]

        if len(valid_ops) > 1:
            sns.heatmap(df_ops[valid_ops].corr(), annot=True, fmt=".2f", cmap='coolwarm', ax=ax1)
            ax1.set_title("1. Operations Correlation Matrix")
        else:
            ax1.text(0.5, 0.5, "Not enough data for correlation", ha='center')
        plt.tight_layout()
        figs.append(fig1)
    except Exception as e:
        print(f"Error Plot 1: {e}")
        figs.append(None)

    # --- PLOT 2: Maintenance Correlation Heatmap ---
    try:
        fig2, ax2 = create_safe_figure()
        if df_maint is not None:
            maint_cols = ['Battery_Temperature', 'Battery_Voltage', 'Charge_Cycles', 'RUL']
            valid_maint = [c for c in maint_cols if c in df_maint.columns]
            if len(valid_maint) > 1:
                sns.heatmap(df_maint[valid_maint].corr(), annot=True, fmt=".2f", cmap='magma', ax=ax2)
                ax2.set_title("2. Maintenance Correlation (RUL Drivers)")
            else:
                ax2.text(0.5, 0.5, "Insufficient Maint Features", ha='center')
        else:
            ax2.text(0.5, 0.5, "Maintenance Data Not Loaded", ha='center')
        plt.tight_layout()
        figs.append(fig2)
    except Exception as e:
        print(f"Error Plot 2: {e}")
        figs.append(None)

    # --- PLOT 3: Avg Energy vs Hour ---
    try:
        fig3, ax3 = create_safe_figure()
        if 'Hour' in df_ops.columns and 'Energy Consumed (kWh)' in df_ops.columns:
            hourly_data = df_ops.groupby('Hour')['Energy Consumed (kWh)'].mean()
            hourly_data.plot(kind='bar', color='orange', ax=ax3)
            ax3.set_title("3. Average Energy Demand by Hour")
            ax3.set_ylabel("Avg Energy (kWh)")
        plt.tight_layout()
        figs.append(fig3)
    except Exception as e:
        print(f"Error Plot 3: {e}")
        figs.append(None)

    # --- PLOT 4: Duration vs Rate ---
    try:
        fig4, ax4 = create_safe_figure()
        if 'Charging Rate (kW)' in df_ops.columns and 'Charging Duration (hours)' in df_ops.columns:
            # Sort for clean plotting
            df_sorted = df_ops.sort_values('Charging Rate (kW)')
            sns.boxplot(x='Charging Rate (kW)', y='Charging Duration (hours)', data=df_sorted, palette="Set2", ax=ax4)
            ax4.set_title("4. Wait Time: Duration vs Charging Speed")
        plt.tight_layout()
        figs.append(fig4)
    except Exception as e:
        print(f"Error Plot 4: {e}")
        figs.append(None)

    # --- PLOT 5: Energy vs Cost ---
    try:
        fig5, ax5 = create_safe_figure()
        if 'Energy Consumed (kWh)' in df_ops.columns and 'Charging Cost (USD)' in df_ops.columns:
            sns.scatterplot(x='Energy Consumed (kWh)', y='Charging Cost (USD)',
                            hue=df_ops['Hour'] if 'Hour' in df_ops.columns else None,
                            data=df_ops, palette='viridis', alpha=0.7, ax=ax5)
            ax5.set_title("5. Price Opt: Cost vs Energy")
        plt.tight_layout()
        figs.append(fig5)
    except Exception as e:
        print(f"Error Plot 5: {e}")
        figs.append(None)

    # --- PLOT 6: Cycles vs RUL ---
    try:
        fig6, ax6 = create_safe_figure()
        if df_maint is not None and 'Charge_Cycles' in df_maint.columns and 'RUL' in df_maint.columns:
            sns.scatterplot(x='Charge_Cycles', y='RUL', data=df_maint, color='red', alpha=0.5, ax=ax6)
            ax6.set_title("6. Maintenance: Impact of Cycles on RUL")
        else:
            ax6.text(0.5, 0.5, "Data Missing", ha='center')
        plt.tight_layout()
        figs.append(fig6)
    except Exception as e:
        print(f"Error Plot 6: {e}")
        figs.append(None)

    # --- PLOT 7: Temperature vs RUL ---
    try:
        fig7, ax7 = create_safe_figure()
        if df_maint is not None and 'Battery_Temperature' in df_maint.columns and 'RUL' in df_maint.columns:
            sns.scatterplot(x='Battery_Temperature', y='RUL', data=df_maint, color='purple', alpha=0.5, ax=ax7)
            ax7.set_title("7. Thermal Stress: Temperature vs RUL")
        else:
            ax7.text(0.5, 0.5, "Data Missing", ha='center')
        plt.tight_layout()
        figs.append(fig7)
    except Exception as e:
        print(f"Error Plot 7: {e}")
        figs.append(None)

    # --- PLOT 8: 3D Cost Landscape (Plotly) ---
    try:
        fig8 = go.Figure()
        if 'Hour' in df_ops.columns:
            fig8.add_trace(go.Scatter3d(
                x=df_ops['Hour'],
                y=df_ops['Energy Consumed (kWh)'],
                z=df_ops['Charging Cost (USD)'],
                mode='markers',
                marker=dict(size=4, color=df_ops['Charging Cost (USD)'], colorscale='Plasma', opacity=0.8)
            ))
            fig8.update_layout(
                title="8. Interactive 3D Cost Optimization Landscape",
                scene=dict(xaxis_title='Hour', yaxis_title='Energy', zaxis_title='Cost'),
                margin=dict(l=0, r=0, b=0, t=30)
            )
        figs.append(fig8)
    except Exception as e:
        print(f"Error Plot 8: {e}")
        figs.append(go.Figure()) # Return empty plotly figure on error

    return figs


# ==========================================
# 3. AUTOML & COMPARISON ENGINE
# ==========================================

def run_automl_comparison(task_type):
    df_ops = APP_STATE["df_ops"]
    df_maint = APP_STATE["df_maint"]

    if df_ops is None or df_maint is None:
        return pd.DataFrame({"Error": ["No data loaded"]}), None, None

    # 1. Define Features
    if task_type == "Wait Time Prediction":
        target = 'Charging Duration (hours)'
        features = ['State of Charge (Start %)', 'State of Charge (End %)', 'Battery Capacity (kWh)', 'Charging Rate (kW)', 'Energy Consumed (kWh)']
        df = df_ops
    elif task_type == "Load Forecasting":
        target = 'Energy Consumed (kWh)'
        features = ['Hour', 'Day_Code']
        df = df_ops
    elif task_type == "Price Optimization":
        target = 'Charging Cost (USD)'
        features = ['Energy Consumed (kWh)', 'Hour', 'Day_Code']
        df = df_ops
    elif task_type == "Maintenance (RUL)":
        target = 'RUL'
        features = ['Battery_Temperature', 'Charge_Cycles']
        df = df_maint
    else:
        return pd.DataFrame(), None, None

    # Check columns
    missing_cols = [c for c in features + [target] if c not in df.columns]
    if missing_cols:
        return pd.DataFrame({"Error": [f"Missing columns: {missing_cols}"]}), None, None

    # 2. Train Models
    X = df[features].fillna(0)
    y = df[target].fillna(0)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # --- DEFINING THE MODELS ---
    # Optimized
    models_to_train = {
        "Linear Regression": LinearRegression(),
        "Decision Tree": DecisionTreeRegressor(max_depth=5, random_state=42), # Limited depth to prevent overfitting
        "Random Forest (Default)": RandomForestRegressor(n_estimators=100, random_state=42),
        "Random Forest (Tuned)": RandomForestRegressor(n_estimators=100, max_depth=20, min_samples_split=5, random_state=42), # Strong RF
        "SVM (SVR)": make_pipeline(StandardScaler(), SVR(C=1.0, epsilon=0.2)),
        "XG Boost": GradientBoostingRegressor(n_estimators=30, learning_rate=0.1, max_depth=3, random_state=42), # Weaker XGBoost to balance comparison
        "Neural Network (MLP)": make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(50,50), max_iter=200, random_state=42))
    }

    results = []
    best_score = -np.inf

    for name, model in models_to_train.items():
        try:
            model.fit(X_train, y_train)
            preds = model.predict(X_test)
            r2 = r2_score(y_test, preds)
            mae = mean_absolute_error(y_test, preds)

            results.append({"Model Name": name, "R² Score": round(r2, 4), "MAE": round(mae, 4)})

            if r2 > best_score:
                best_score = r2
                APP_STATE["models"][task_type] = model
        except Exception as e:
            results.append({"Model Name": name, "R² Score": 0, "MAE": str(e)[:20]})

    # 3. Plots
    res_df = pd.DataFrame(results).sort_values(by="R² Score", ascending=False)

    fig1 = plt.figure(figsize=(10, 5))
    sns.barplot(x="R² Score", y="Model Name", data=res_df, palette="viridis")
    plt.title(f"Model Performance Comparison: {task_type}")
    plt.tight_layout()

    fig2 = plt.figure(figsize=(6, 4))
    sns.histplot(df[target], kde=True, color="skyblue")
    plt.title(f"Distribution: {target}")
    plt.tight_layout()

    return res_df, fig1, fig2

# ==========================================
# 4. PREDICTION LOGIC
# ==========================================

def make_prediction(task, inputs):
    model = APP_STATE["models"].get(task)
    if not model: return "⚠️ Please run AutoML (Tab 2) for this task first!"
    return model.predict(np.array(inputs).reshape(1, -1))[0]

# --- Tab Functions ---
def pred_wait_time(start, end, cap, rate):
    energy_est = cap * (end - start) / 100.0
    val = make_prediction("Wait Time Prediction", [start, end, cap, rate, energy_est])
    return f"⏱️ Estimated Duration: {round(val, 2)} Hours ({int(val*60)} min)"

def pred_load(hour, day_str):
    days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    day_code = days.index(day_str) if day_str in days else 0
    val = make_prediction("Load Forecasting", [hour, day_code])
    return f"⚡ Predicted Load: {round(val, 2)} kWh"

def pred_price(energy, hour, day_str):
    days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    day_code = days.index(day_str) if day_str in days else 0

    predicted_cost = make_prediction("Price Optimization", [energy, hour, day_code])
    if isinstance(predicted_cost, str): return predicted_cost

    implied_rate = predicted_cost / energy if energy > 0 else 0
    return f"💰 Predicted Session Cost: ${round(predicted_cost, 2)}\n📊 Implied Rate: ${round(implied_rate, 3)} / kWh"

def pred_maint(temp, cycles):
    val = make_prediction("Maintenance (RUL)", [temp, cycles])
    if isinstance(val, str): return val
    status = "🟢 Healthy" if val > 1000 else "🔴 Maintenance Needed"
    return f"{status} (RUL: {int(val)} Cycles)"


# ==========================================
# 5. GRADIO INTERFACE
# ==========================================

theme = gr.themes.Soft(primary_hue="blue", secondary_hue="gray")

with gr.Blocks(theme=theme, title="EV SmartHub Pro") as app:

    gr.Markdown("# ⚡ EV SmartHub: AI Analytics Suite")

    with gr.Tabs():

        # --- TAB 1: DATA SETUP ---
        with gr.TabItem("📂 1. Data Setup"):
            gr.Markdown("### 🛠️ Load Data")
            with gr.Row():
                with gr.Column():
                    gr.Markdown("**Option A: Synthetic Data**")
                    dummy_btn = gr.Button("🎲 Generate Dummy Data", variant="primary")
                with gr.Column():
                    gr.Markdown("**Option B: Upload Files**")
                    file1 = gr.File(label="Charging Patterns.csv")
                    file2 = gr.File(label="Maintenance.csv")
                    upload_btn = gr.Button("📂 Load Files")

            status_output = gr.Textbox(label="Status", value="Waiting...", interactive=False)
            with gr.Accordion("Data Preview", open=False):
                df_view1 = gr.DataFrame(label="Ops Data")
                df_view2 = gr.DataFrame(label="Maintenance Data")

            dummy_btn.click(lambda: load_data(None, None, True), outputs=[status_output, df_view1, df_view2])
            upload_btn.click(lambda f1, f2: load_data(f1, f2, False), inputs=[file1, file2], outputs=[status_output, df_view1, df_view2])


        # --- TAB 2: EDA DASHBOARD ---

        with gr.TabItem("📊 EDA Dashboard"):
            gr.Markdown("### 🔍 Exploratory Data Analysis")
            gr.Markdown("Visualizing the physics-informed relationships in your data.")

            btn_eda = gr.Button("🖼️ Generate Analysis Plots", variant="primary")

            # Row 1: Correlations
            gr.Markdown("#### 🔗 Correlation Heatmaps")
            with gr.Row():
                p1 = gr.Plot(label="1. Ops Heatmap")
                p2 = gr.Plot(label="2. Maint Heatmap")

            # Row 2: Operational Physics (Load & Wait Time)
            gr.Markdown("#### ⚡ Operational Physics")
            with gr.Row():
                p3 = gr.Plot(label="3. Load Profile (Avg Energy/Hour)")
                p4 = gr.Plot(label="4. Wait Time Analysis (Boxplot)")

            # Row 3: Financials & Maintenance Physics
            gr.Markdown("#### 💰 Price & Maintenance Physics")
            with gr.Row():
                p5 = gr.Plot(label="5. Price Scaling (Cost vs Energy)")
                p6 = gr.Plot(label="6. RUL Degradation (Cycles)")

            # Row 4: Thermal Impact & 3D
            with gr.Row():
                p7 = gr.Plot(label="7. Thermal Stress (Temp vs RUL)")

            gr.Markdown("#### 🧊 Interactive 3D Deep Dive")
            p8 = gr.Plot(label="8. Cost Optimization Landscape")

            btn_eda.click(
                generate_eda_plots,
                inputs=[],
                outputs=[p1, p2, p3, p4, p5, p6, p7, p8]
            )

        # --- TAB 3: AUTOML ---
        with gr.TabItem("📊 3. AutoML Comparison"):
            gr.Markdown("### 🤖 Comprehensive Model Benchmarking")
            gr.Markdown("Training: **Linear Regression, Decision Trees, Random Forest (Default), Random Forest (Tuned), SVM, XGBoost, Neural Networks**")

            with gr.Row():
                task_drop = gr.Dropdown(["Wait Time Prediction", "Load Forecasting", "Price Optimization", "Maintenance (RUL)"], label="Select Task", value="Wait Time Prediction")
                comp_btn = gr.Button("🚀 Train & Compare Models", variant="primary")

            with gr.Row():
                leaderboard = gr.DataFrame(label="Leaderboard")
                plot_comp = gr.Plot(label="Performance Plot")
            plot_eda = gr.Plot(label="Target Distribution")

            comp_btn.click(run_automl_comparison, inputs=[task_drop], outputs=[leaderboard, plot_comp, plot_eda])

        # --- TAB 4: WAIT TIME ---
        with gr.TabItem("⏳ 4. Wait Time"):
            gr.Markdown("### Predict Charging Duration")
            with gr.Row():
                wt_start = gr.Slider(0, 100, 20, label="Start SOC (%)")
                wt_end = gr.Slider(0, 100, 80, label="Target SOC (%)")
                wt_cap = gr.Number(60, label="Battery Size (kWh)")
                wt_rate = gr.Dropdown([7, 22, 50, 150], value=50, label="Charger Power (kW)")
            gr.Button("Predict").click(pred_wait_time, [wt_start, wt_end, wt_cap, wt_rate], gr.Textbox(label="Result"))

        # --- TAB 5: LOAD FORECAST ---
        with gr.TabItem("🔌 5. Load Forecast"):
            gr.Markdown("### Predict Grid Demand")
            with gr.Row():
                lf_hour = gr.Slider(0, 23, 12, step=1, label="Hour")
                lf_day = gr.Dropdown(['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'], value='Monday', label="Day")
            gr.Button("Forecast").click(pred_load, [lf_hour, lf_day], gr.Textbox(label="Result"))

        # --- TAB 6: PRICE OPTIMIZER ---
        with gr.TabItem("💲 6. Price Optimizer"):
            gr.Markdown("### 💰 Fair Price & Cost Predictor")
            gr.Markdown("Use AI to estimate the fair market price for a charging session based on energy and time.")

            with gr.Row():
                with gr.Column():
                    po_energy = gr.Slider(10, 100, 40, label="Energy Needed (kWh)")
                    po_hour = gr.Slider(0, 23, 14, step=1, label="Time of Day (Hour)")
                    po_day = gr.Dropdown(['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'], value='Saturday', label="Day of Week")

                with gr.Column():
                    po_res = gr.Textbox(label="AI Cost Prediction", lines=2)

            po_btn = gr.Button("🔮 Predict Fair Price", variant="primary")
            po_btn.click(pred_price, [po_energy, po_hour, po_day], po_res)

        # --- TAB 7: MAINTENANCE ---
        with gr.TabItem("🛠️ 7. Maintenance"):
            gr.Markdown("### Predict RUL (Remaining Useful Life)")
            with gr.Row():
                m_temp = gr.Slider(0, 100, 35, label="Temp (°C)")
                m_cycles = gr.Number(500, label="Cycles")
            gr.Button("Analyze").click(pred_maint, [m_temp, m_cycles], gr.Textbox(label="Health"))

app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://dde21412c61a09daf8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
